# Баланс урона, здоровья и TTK

Этот ноутбук последовательно описывает фундаментальные правила боевого баланса. Кодовые ячейки содержат только формулы игровой логики и короткие проверяемые примеры.

Все расчёты используют соответствующие друг другу тир противника, тир оружия и уровень игрока.

Глобальный переключатель `UPDATE_CSV` управляет всеми тремя шагами записи в конце ноутбука. По умолчанию он выключен. Установите `True`, выполните весь ноутбук для обновления `Weapon.csv`, `Armor.csv` и `Enemies.csv`, затем верните `False` перед коммитом.

In [2]:
UPDATE_CSV = False

## 1. TTK как фундамент баланса

TTK, или time to kill, показывает, сколько **успешных попаданий** требуется, чтобы убить цель. Промахи, длительность хода, дальность и перемещение в само значение TTK не входят.

```text
TTK = hit_points / effective_damage_per_hit
```

Для настройки используется непрерывное отношение. Фактическое количество попаданий до смерти округляется вверх: результат `3.2` означает смерть от четвёртого успешного попадания.

Диапазоны TTK не растут вместе с тиром. Конкретное значение внутри диапазона зависит от профиля противника, оружия и защиты, а среднее используется как отправная точка расчётов:

| Цель | Диапазон TTK | Средний TTK | Смысл |
| --- | ---: | ---: | --- |
| Противник | `2 .. 5` | `3` | Разные профили равного тира выдерживают разное количество попаданий средним оружием |
| Игрок | `12 .. 30` | `20` | Разные противники равного тира создают разное давление на игрока |

Границы задают ожидаемую область обычных боёв, а не обязательный предел для уникальных противников. Среднее значение не обязано совпадать с серединой диапазона: большинство обычных врагов находится ближе к нему, а крайние значения предназначены для выраженных профилей.

In [3]:
from math import ceil
from decimal import Decimal, ROUND_HALF_UP

ENEMY_TTK_RANGE = (2, 5)
PLAYER_TTK_RANGE = (12, 30)
AVERAGE_ENEMY_TTK = 3
AVERAGE_PLAYER_TTK = 20

def effective_damage(raw_damage: float, armor: float) -> float:
    """Урон успешного попадания после плоского вычитания брони."""
    return max(0.0, raw_damage - armor)

def ttk_ratio(hit_points: float, damage_per_hit: float) -> float:
    """Непрерывный TTK для балансировки."""
    if damage_per_hit <= 0:
        raise ValueError("Урон должен быть больше нуля для расчёта TTK")
    return hit_points / damage_per_hit

def hits_to_kill(hit_points: float, damage_per_hit: float) -> int:
    """Целое количество успешных попаданий до смерти."""
    return ceil(ttk_ratio(hit_points, damage_per_hit))

def round_half_up(value: float) -> int:
    """Округление до ближайшего целого; точная половина идёт вверх."""
    return int(Decimal(str(value)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))

assert ttk_ratio(30, 10) == 3
assert hits_to_kill(31, 10) == 4

## 2. Прогрессия урона оружия

Эталон первого тира задан интуитивно: `4` среднего урона для оружия ближнего боя с обычной скоростью атаки. Каждый следующий тир умножает эталон на постоянный коэффициент `1.5`. Геометрическая прогрессия делает правило явным и сохраняет ощутимый, но контролируемый прирост силы.

```text
reference_weapon_damage(tier) = 4 x 1.5^(tier - 1)
```

Диапазон в таблице рассчитан для обычного оружия ближнего боя без `VAR` и индивидуальных коэффициентов.

| Тир | Диапазон урона | Эталонный средний урон | Рост к предыдущему тиру |
| ---: | ---: | ---: | ---: |
| 1 | `2 .. 6` | `4.00` | — |
| 2 | `4 .. 8` | `6.00` | `x1.5` |
| 3 | `6 .. 12` | `9.00` | `x1.5` |
| 4 | `9 .. 19` | `13.50` | `x1.5` |
| 5 | `13 .. 27` | `20.25` | `x1.5` |
| 6 | `20 .. 40` | `30.38` | `x1.5` |
| 7 | `31 .. 61` | `45.56` | `x1.5` |
| 8 | `45 .. 91` | `68.34` | `x1.5` |

In [4]:
TIER_1_REFERENCE_WEAPON_DAMAGE = 4.0
WEAPON_TIER_DAMAGE_GROWTH = 1.5

def reference_weapon_damage(tier: int) -> float:
    """Средний урон обычного оружия соответствующего тира."""
    if tier not in range(1, 9):
        raise ValueError("Тир оружия должен находиться в диапазоне 1..8")
    return TIER_1_REFERENCE_WEAPON_DAMAGE * WEAPON_TIER_DAMAGE_GROWTH ** (tier - 1)

assert reference_weapon_damage(1) == 4.0
assert round(reference_weapon_damage(5), 2) == 20.25

## 3. Профиль оружия

`STR` и `INT` только выбирают характеристику для формулы боевого урона. `DEX`-оружие атакует за `8 mp` вместо обычных `10 mp`, поэтому его урон за попадание умножается на `0.8`. Это компенсирует частоту атак `1.25x`.

Каждая единица дальности после первой умножает урон на `0.85`. Дополнительная дистанция не гарантирует ещё одно попадание, а дальнобойное оружие требует боеприпасы.

```text
target_weapon_damage =
    reference_weapon_damage
    x attack_speed_factor
    x 0.85^(range - 1)
    x exceptional_weapon_factor
```

Обычный диапазон урона имеет половину ширины около `33%` от округлённого среднего, но не меньше `2`. Благодаря этому диапазоны соседних тиров пересекаются. Trait `VAR` (`variable damage`) обозначает оружие с широким и менее предсказуемым диапазоном: для него половина ширины равна `50%`. Trait меняет только разброс, сохраняя средний урон.

Два оружия имеют явные коэффициенты-исключения:

| Оружие | Коэффициент | Причина |
| --- | ---: | --- |
| `Torch` | `0.5` (`2 / 4`) | Средний урон `2` сохранён из диапазона `1 .. 3`; основная ценность факела — свет и огонь |
| `Tesla Rod` | `1.10` | Редкое древнее оружие получает на `10%` больше среднего урона |

Огонь, кислота и яд являются бонусами и не уменьшают базовый урон.

In [5]:
DEX_ATTACK_DAMAGE_FACTOR = 0.8
RANGE_DAMAGE_FACTOR = 0.85
NORMAL_DAMAGE_SPREAD = 1 / 3
VARIABLE_DAMAGE_SPREAD = 0.50
MINIMUM_DAMAGE_HALF_WIDTH = 2
WEAPON_DAMAGE_FACTOR = {"Torch": 0.5, "Tesla Rod": 1.10}
WEAPON_FIXED_DAMAGE_RANGE = {"Torch": (1, 3)}

def weapon_target_damage(tier: int, stat: str, attack_range: int, weapon_name: str | None = None) -> float:
    """Целевой средний урон с учётом скорости, дальности и исключений."""
    speed_factor = DEX_ATTACK_DAMAGE_FACTOR if stat == "DEX" else 1.0
    range_factor = RANGE_DAMAGE_FACTOR ** (attack_range - 1)
    exceptional_factor = WEAPON_DAMAGE_FACTOR.get(weapon_name, 1.0)
    return reference_weapon_damage(tier) * speed_factor * range_factor * exceptional_factor

def weapon_damage_range(target_damage: float, variable: bool = False, weapon_name: str | None = None) -> tuple[int, int]:
    """Целочисленный диапазон вокруг округлённого среднего урона."""
    if weapon_name in WEAPON_FIXED_DAMAGE_RANGE:
        return WEAPON_FIXED_DAMAGE_RANGE[weapon_name]
    center = round_half_up(target_damage)
    spread = VARIABLE_DAMAGE_SPREAD if variable else NORMAL_DAMAGE_SPREAD
    half_width = max(MINIMUM_DAMAGE_HALF_WIDTH, round_half_up(center * spread))
    return max(1, center - half_width), center + half_width

assert weapon_target_damage(3, "STR", 1) == 9
assert round(weapon_target_damage(3, "DEX", 3), 3) == 5.202
assert weapon_target_damage(1, "STR", 1, "Torch") == 2
assert weapon_damage_range(2, weapon_name="Torch") == (1, 3)
assert weapon_damage_range(4) == (2, 6)
assert weapon_damage_range(6) == (4, 8)
assert weapon_damage_range(10, variable=True) == (5, 15)

## 4. Эталонный игрок

Уровень игрока принимается равным тиру противника.

```text
player_level = tier
player_HP = 20 + level x 6 x (1 + 0.15 x health)
```

Базовый расчёт использует `health = 0` и предполагает `100%` попаданий. Владение оружием в TTK не участвует.

In [6]:
PLAYER_HP_BASE = 20
PLAYER_HP_PER_LEVEL = 6
PLAYER_HEALTH_FACTOR = 0.15

def player_hit_points(level: int, health: int = 0) -> float:
    """Максимальные HP игрока для заданного уровня и здоровья."""
    return PLAYER_HP_BASE + level * PLAYER_HP_PER_LEVEL * (1 + PLAYER_HEALTH_FACTOR * health)

assert [player_hit_points(t) for t in (1, 3, 5, 8)] == [26, 38, 50, 68]

## 5. Урон игрока

Урон успешного попадания зависит от оружия, силы и соответствующей оружию характеристики:

```text
player_damage =
    average_weapon_damage
    x (1 + 0.05 x STR + 0.10 x matching_stat)
```

Для нейтрального персонажа обе характеристики равны нулю, поэтому урон равен среднему урону оружия.

In [7]:
STRENGTH_DAMAGE_FACTOR = 0.05
MATCHING_STAT_DAMAGE_FACTOR = 0.10

def player_damage_per_hit(average_weapon_damage: float, strength: int = 0, matching_stat: int = 0) -> float:
    """Урон успешного попадания игрока до брони цели."""
    multiplier = 1 + STRENGTH_DAMAGE_FACTOR * strength + MATCHING_STAT_DAMAGE_FACTOR * matching_stat
    return average_weapon_damage * multiplier

assert player_damage_per_hit(16) == 16
assert player_damage_per_hit(16, strength=2, matching_stat=2) == 20.8

## 6. Броня противника

Числовая броня есть только у противников с `ARM` и поглощает `20%` среднего урона оружия того же тира:

```text
enemy_armor = round_half_up(tier_average_weapon_damage x 0.20)
```

Без `ARM` броня равна нулю. Числовое значение брони и особенность `ARM` всегда должны совпадать.

In [8]:
ENEMY_ARMOR_DAMAGE_SHARE = 0.20

def enemy_armor(tier_average_weapon_damage: float, armored: bool) -> int:
    """Броня среднего ARM-противника соответствующего тира."""
    if not armored:
        return 0
    return round_half_up(tier_average_weapon_damage * ENEMY_ARMOR_DAMAGE_SHARE)

assert enemy_armor(25, armored=True) == 5
assert enemy_armor(25, armored=False) == 0

## 7. HP противника

При расчёте HP броня компенсирует только половину своей силы. Она не должна полностью заменять здоровье: иначе два противника с одинаковым профилем, но разным сочетанием HP и брони имели бы практически одинаковую живучесть.

```text
armor_HP_compensation = 0.5
HP_damage_basis = max(0, player_damage - enemy_armor x armor_HP_compensation)
base_enemy_HP = enemy_TTK x HP_damage_basis
```

Коэффициент `0.5` означает, что половина защиты брони учитывается уменьшением HP, а вторая половина остаётся дополнительной живучестью бронированного противника. В самом бою правило не меняется: из входящего урона вычитается полное значение брони. Поэтому средний `enemy_TTK = 3` является базовой точкой для небронированного врага, а бронированный враг выдерживает немного больше успешных попаданий.

Например, при уроне игрока `25`, броне `5` и целевом TTK `3` противник получает `68 HP` после округления. Фактический урон после брони равен `20`, поэтому непрерывный TTK составляет `68 / 20 = 3.4`. При полной компенсации брони его HP были бы равны `60`, а TTK снова стал бы ровно `3`, то есть броня не давала бы отдельного преимущества.

Ловкость и скорость передвижения меняют итоговые HP, потому что сами повышают или понижают опасность врага.

| Особенность | Множитель HP |
| --- | ---: |
| `DEX+` | `0.88` |
| `DEX-` | `1.12` |
| `SPD+` | `0.90` |
| `SPD-` | `1.10` |
| обычная | `1.00` |

In [9]:
ARMOR_HP_COMPENSATION = 0.5
DEXTERITY_HP_FACTOR = {"DEX+": 0.88, "DEX-": 1.12}
MOVEMENT_HP_FACTOR = {"SPD+": 0.90, "SPD-": 1.10}

def enemy_hit_points(player_damage: float, armor: int = 0, dexterity_trait: str | None = None, movement_trait: str | None = None, target_ttk: float = AVERAGE_ENEMY_TTK) -> int:
    """HP врага из целевого TTK и его защитного профиля."""
    damage_basis = max(0.0, player_damage - armor * ARMOR_HP_COMPENSATION)
    dexterity = DEXTERITY_HP_FACTOR.get(dexterity_trait, 1.0)
    movement = MOVEMENT_HP_FACTOR.get(movement_trait, 1.0)
    return round_half_up(target_ttk * damage_basis * dexterity * movement)

armored_hp = enemy_hit_points(25, armor=5)
actual_ttk = ttk_ratio(armored_hp, effective_damage(25, 5))

assert armored_hp == 68
assert actual_ttk == 3.4
assert enemy_hit_points(25, armor=5, dexterity_trait="DEX+") == 59

## 8. Броня игрока

Броня растёт по двум правилам одновременно. Она поглощает около `20%` эталонного среднего урона оружия того же тира, но её центр не может быть меньше номера тира. Минимум по тиру предотвращает одинаковые значения в начале игры, где `20%` небольшого урона после округления почти не меняются. Начиная с поздних тиров рост урона естественно обгоняет линейный минимум.

```text
player_armor_center = max(
    tier,
    round_half_up(reference_weapon_damage x 0.20)
)
armor_half_width = max(1, round_half_up(player_armor_center x 0.20))
```

| Тир | Диапазон | Центр брони |
| ---: | ---: | ---: |
| 1 | `1 .. 2` | `1` |
| 2 | `1 .. 3` | `2` |
| 3 | `2 .. 4` | `3` |
| 4 | `3 .. 5` | `4` |
| 5 | `4 .. 6` | `5` |
| 6 | `5 .. 7` | `6` |
| 7 | `7 .. 11` | `9` |
| 8 | `11 .. 17` | `14` |

In [10]:
PLAYER_ARMOR_DAMAGE_SHARE = 0.20
PLAYER_ARMOR_SPREAD = 0.20
def player_armor_center(tier: int) -> int:
    """Центр диапазона брони игрока соответствующего тира."""
    damage_based = round_half_up(reference_weapon_damage(tier) * PLAYER_ARMOR_DAMAGE_SHARE)
    return max(tier, damage_based)

def player_armor_range(tier: int) -> tuple[int, int]:
    """Целочисленный диапазон защиты доспеха соответствующего тира."""
    center = player_armor_center(tier)
    half_width = max(1, round_half_up(center * PLAYER_ARMOR_SPREAD))
    return max(1, center - half_width), center + half_width

def average_player_armor(tier: int) -> float:
    """Средняя броня тира без индивидуального смещения предмета."""
    return float(player_armor_center(tier))

assert player_armor_range(1) == (1, 2)
assert player_armor_range(2) == (1, 3)
assert player_armor_range(5) == (4, 6)
assert player_armor_range(8) == (11, 17)

## 9. Урон противника

Сначала из HP игрока и постоянного TTK вычисляется урон, который должен пройти через броню. Затем скорость атаки и дальность уменьшают урон одного попадания. Эффекты остаются бонусами.

Расчёт не требует от игрока уже носить броню текущего тира. Для противника тира `N` используется средняя броня предыдущего тира `N - 1`; для первого тира броня равна `0`. Поэтому подходящая текущему этажу броня даёт игроку преимущество, а не является обязательным условием достижения целевого TTK.

Эталонное значение берётся из средней брони предыдущего тира, рассчитанной в разделе 8.

```text
effective_enemy_damage = player_HP / player_TTK

attack_speed_factor = 0.8 for ATK+, 1.0 normally, 1.2 for ATK-
range_factor = 0.85^(range - 1)

raw_enemy_damage =
    previous_tier_player_armor
    + effective_enemy_damage x attack_speed_factor x range_factor
```

Эталонная броня предыдущего тира добавляется после коэффициентов. Благодаря этому быстрые и дальнобойные атаки не оказываются полностью заблокированы бронёй игрока. Фактически надетая броня применяется уже в бою и может отличаться от эталона.

In [11]:
ATTACK_DAMAGE_FACTOR = {"ATK+": 0.8, "ATK-": 1.2}
ENEMY_DAMAGE_SPREAD = 0.20

def previous_tier_player_armor(enemy_tier: int) -> float:
    """Эталонная броня игрока: предыдущий тир или 0 для начала игры."""
    if enemy_tier == 1:
        return 0.0
    return average_player_armor(enemy_tier - 1)

def enemy_damage_per_hit(player_hp: float, enemy_tier: int, attack_range: int = 1, attack_trait: str | None = None, target_ttk: float = AVERAGE_PLAYER_TTK) -> float:
    """Средний сырой урон успешного попадания врага."""
    damage_after_armor = player_hp / target_ttk
    speed_factor = ATTACK_DAMAGE_FACTOR.get(attack_trait, 1.0)
    range_factor = RANGE_DAMAGE_FACTOR ** (attack_range - 1)
    armor_reference = previous_tier_player_armor(enemy_tier)
    return armor_reference + damage_after_armor * speed_factor * range_factor

def enemy_damage_range(target_damage: float) -> tuple[int, int]:
    """Узкий целочисленный диапазон вокруг среднего урона врага."""
    center = round_half_up(target_damage)
    half_width = max(1, round_half_up(center * ENEMY_DAMAGE_SPREAD))
    return max(1, center - half_width), center + half_width

assert previous_tier_player_armor(1) == 0
assert previous_tier_player_armor(6) == 5
assert enemy_damage_per_hit(50, enemy_tier=6) == 7.5
assert round(enemy_damage_per_hit(50, enemy_tier=6, attack_range=3), 3) == 6.806
assert enemy_damage_range(7.5) == (6, 10)

## 10. Последовательность расчёта

1. Берём средний урон оружия того же тира.
2. По `ARM` определяем броню противника.
3. Выбираем целевой TTK противника в диапазоне `2 .. 5`; для среднего профиля используем `3`.
4. Вычитаем из урона половину брони и из полученной основы и выбранного TTK вычисляем базовые HP.
5. Корректируем HP через `DEX` и скорость передвижения.
6. Приравниваем уровень игрока к тиру и вычисляем HP игрока.
7. Выбираем целевой TTK игрока в диапазоне `12 .. 30`; для среднего профиля используем `20`.
8. Из HP игрока и выбранного TTK получаем урон после брони.
9. Корректируем урон врага через скорость атаки и дальность.
10. Добавляем среднюю броню предыдущего тира и получаем сырой средний урон врага; для первого тира добавляем `0`.

Диапазоны и средние TTK остаются постоянными. С тиром растут урон оружия, HP игрока, броня и абсолютные характеристики противников.

## 11. Обновление `Weapon.csv`

Эта ячейка пересчитывает диапазон урона каждого оружия по формулам ноутбука. Она читает тир, характеристику, дальность, имя и отдельную колонку `Trait`, после чего меняет только колонку `Damage`. Значение `VAR` в `Trait` включает широкий разброс; колонка `Effect` остаётся только для элементальных эффектов.

Запись выполняется только когда глобальный переключатель `UPDATE_CSV` установлен в `True`.

In [12]:
from pathlib import Path
import csv

if UPDATE_CSV:
    # Первый путь используется из корня репозитория, второй — из misc/tools.
    candidates = [Path("misc/entities/Weapon.csv"), Path("../entities/Weapon.csv")]
    weapon_csv_path = next((path for path in candidates if path.exists()), None)
    if weapon_csv_path is None:
        raise FileNotFoundError("Не найден misc/entities/Weapon.csv")

    with weapon_csv_path.open(encoding="utf-8", newline="") as source:
        reader = csv.DictReader(source, delimiter=";")
        fieldnames = reader.fieldnames
        weapons = list(reader)

    for weapon in weapons:
        target_damage = weapon_target_damage(
            tier=int(weapon["Tier"]),
            stat=weapon["Stat"],
            attack_range=int(weapon["Range"]),
            weapon_name=weapon["Name"],
        )
        traits = {trait.strip() for trait in weapon["Trait"].split(",") if trait.strip()}
        minimum, maximum = weapon_damage_range(
            target_damage, variable="VAR" in traits, weapon_name=weapon["Name"]
        )
        weapon["Damage"] = f"{minimum}-{maximum}"

    with weapon_csv_path.open("w", encoding="utf-8", newline="") as destination:
        writer = csv.DictWriter(destination, fieldnames=fieldnames, delimiter=";", lineterminator="\n")
        writer.writeheader()
        writer.writerows(weapons)

    print(f"Updated {len(weapons)} weapons in {weapon_csv_path}")

Updated 40 weapons in ../entities/Weapon.csv


## 12. Обновление `Armor.csv`

Эта ячейка пересчитывает колонку `Armor` по тиру доспеха. Все границы диапазона целочисленные. Остальные колонки не меняются.

Запись выполняется только когда глобальный переключатель `UPDATE_CSV` установлен в `True`.

In [13]:
if UPDATE_CSV:
    candidates = [Path("misc/entities/Armor.csv"), Path("../entities/Armor.csv")]
    armor_csv_path = next((path for path in candidates if path.exists()), None)
    if armor_csv_path is None:
        raise FileNotFoundError("Не найден misc/entities/Armor.csv")

    with armor_csv_path.open(encoding="utf-8", newline="") as source:
        reader = csv.DictReader(source, delimiter=";")
        fieldnames = reader.fieldnames
        armors = list(reader)

    for armor in armors:
        minimum, maximum = player_armor_range(int(armor["Peak"]))
        armor["Armor"] = f"{minimum}–{maximum}"

    with armor_csv_path.open("w", encoding="utf-8", newline="") as destination:
        writer = csv.DictWriter(destination, fieldnames=fieldnames, delimiter=";", lineterminator="\n")
        writer.writeheader()
        writer.writerows(armors)

    print(f"Updated {len(armors)} armors in {armor_csv_path}")

Updated 8 armors in ../entities/Armor.csv


## 13. Обновление `Enemies.csv`

Эта ячейка пересчитывает три боевые характеристики каждого противника:

- `Броня` зависит от тира и наличия `ARM`; без `ARM` поле остаётся пустым, что означает нулевую броню.
- `HP` зависит от среднего урона оружия того же тира, брони и особенностей `DEX±` и `SPD±`.
- `Урон` зависит от HP игрока соответствующего уровня, средней брони предыдущего тира, `ATK±` и `RNG(n)`. Диапазон имеет разброс `±20%`, но не меньше единицы в каждую сторону.

Элементальные эффекты и сопротивления не меняют базовые числа. Все записываемые значения целочисленные.

Запись выполняется только когда глобальный переключатель `UPDATE_CSV` установлен в `True`.

In [14]:
import re

if UPDATE_CSV:
    candidates = [Path("misc/entities/Enemies.csv"), Path("../entities/Enemies.csv")]
    enemies_csv_path = next((path for path in candidates if path.exists()), None)
    if enemies_csv_path is None:
        raise FileNotFoundError("Не найден misc/entities/Enemies.csv")

    with enemies_csv_path.open(encoding="utf-8", newline="") as source:
        reader = csv.DictReader(source, delimiter=";")
        fieldnames = reader.fieldnames
        enemies = list(reader)

    for enemy in enemies:
        tier = int(enemy["Tier"])
        traits = enemy["Особенности"]
        armored = "ARM" in traits
        armor = enemy_armor(reference_weapon_damage(tier), armored)

        dexterity_match = re.search(r"DEX[+-]", traits)
        movement_match = re.search(r"SPD[+-]", traits)
        attack_match = re.search(r"ATK[+-]", traits)
        range_match = re.search(r"RNG\((\d+)\)", traits)

        enemy["Броня"] = str(armor) if armored else ""
        enemy["HP"] = str(enemy_hit_points(
            player_damage=reference_weapon_damage(tier),
            armor=armor,
            dexterity_trait=dexterity_match.group(0) if dexterity_match else None,
            movement_trait=movement_match.group(0) if movement_match else None,
        ))

        average_damage = enemy_damage_per_hit(
            player_hp=player_hit_points(tier),
            enemy_tier=tier,
            attack_range=int(range_match.group(1)) if range_match else 1,
            attack_trait=attack_match.group(0) if attack_match else None,
        )
        minimum, maximum = enemy_damage_range(average_damage)
        enemy["Урон"] = f"{minimum}–{maximum}"

    with enemies_csv_path.open("w", encoding="utf-8", newline="") as destination:
        writer = csv.DictWriter(destination, fieldnames=fieldnames, delimiter=";", lineterminator="\n")
        writer.writeheader()
        writer.writerows(enemies)

    print(f"Updated {len(enemies)} enemies in {enemies_csv_path}")

Updated 20 enemies in ../entities/Enemies.csv
